In [3]:
from wav_to_embedding import * 
import tqdm

In [11]:
data_directory = "unit_test_data"

wav_files = get_wav_files(data_directory)

hubert_embeddings = []
for file_path, label in tqdm.tqdm(wav_files, desc="Processing WAV files"):
    # Obtain the embedding for the current WAV file
    embedding_tensor = wav_to_embedding(file_path)
    
    # Flatten the embedding tensor
    embedding = embedding_tensor.detach().numpy().flatten()  # Convert tensor to numpy array
    
    # Append the file name, label, and flattened embedding to the data list
    hubert_embeddings.append((file_path, label, embedding))

# Convert the data list to a DataFrame
hubert_embeddings_df = pd.DataFrame(hubert_embeddings, columns=['Soundtrack', 'Label', 'Embedding'])

# Save DataFrame to CSV
hubert_embeddings_df.to_csv('hubert_embeddings.csv', index=False)

Processing WAV files: 100%|██████████████████████| 4/4 [00:00<00:00,  5.90it/s]


In [12]:
hubert_embeddings_df

,Soundtrack,Label,Embedding
0,unit_test_data/Pressed/A3_A_pressedta_norm.wav,Pressed,"[0.19391328, 0.5167813, 0.13981955, 0.17416999..."
1,unit_test_data/Breathy/A3_A_breathy_2_norm.wav,Breathy,"[0.085249886, 0.4596952, 0.3463359, 0.1114618,..."
2,unit_test_data/Normal/A3_A_neutral_norm.wav,Normal,"[0.1342236, 0.61969876, 0.24281624, 0.11822165..."
3,unit_test_data/Flow/A3_A_flow_norm.wav,Flow,"[0.13961063, 0.6294041, 0.21230346, 0.1959945,..."


In [13]:
data_directory = "data"

wav_files = get_wav_files(data_directory)

hubert_embeddings = []
for file_path, label in tqdm.tqdm(wav_files, desc="Processing WAV files"):
    # Obtain the embedding for the current WAV file
    embedding_tensor = wav_to_embedding(file_path)
    
    # Flatten the embedding tensor
    embedding = embedding_tensor.detach().numpy().flatten()  # Convert tensor to numpy array
    
    # Append the file name, label, and flattened embedding to the data list
    hubert_embeddings.append((file_path, label, embedding))

# Convert the data list to a DataFrame
hubert_embeddings_df = pd.DataFrame(hubert_embeddings, columns=['Soundtrack', 'Label', 'Embedding'])

Processing WAV files: 100%|██████████████████| 762/762 [01:58<00:00,  6.42it/s]


In [16]:
phonation_mode_df = hubert_embeddings_df.copy(deep=True)

In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Step 1: Extract features and labels

# X = np.stack(phonation_mode_df['Embedding'])  # Convert string representation of array to numpy array
# y = phonation_mode_df['Label']

# Pad or truncate embeddings to a fixed length
max_length = max(len(embedding) for embedding in phonation_mode_df['Embedding'])
X = np.array([np.pad(embedding, (0, max_length - len(embedding))) for embedding in phonation_mode_df['Embedding']])

# Extract labels
y = phonation_mode_df['Label']

In [19]:
# Step 2: Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3: Train SVM classifier
svm_classifier = SVC(kernel='linear')  # Linear kernel works well for high-dimensional data
svm_classifier.fit(X_train, y_train)

SVC(kernel='linear')

In [20]:
# Step 4: Evaluate classifier
y_pred = svm_classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.8169934640522876


In [21]:
# Create a DataFrame to store results
results_df = pd.DataFrame({
    'Actual_Label': y_test,
    'Predicted_Label': y_pred,
    'Features': X_test.tolist()
})

In [31]:
results_df

,Actual_Label,Predicted_Label,Features
196,Pressed,Pressed,"[0.21948839724063873, 0.7289494872093201, 0.16..."
260,Pressed,Pressed,"[0.20392000675201416, 0.5029839277267456, 0.06..."
39,Pressed,Pressed,"[0.1743122786283493, 0.5102345943450928, 0.068..."
449,Breathy,Breathy,"[0.2533986270427704, 0.49114271998405457, 0.15..."
595,Normal,Normal,"[0.20222653448581696, 0.5663732290267944, 0.13..."
...,...,...,...
550,Normal,Normal,"[0.20668986439704895, 0.5305796265602112, 0.16..."
382,Breathy,Breathy,"[0.259615033864975, 0.49446117877960205, 0.204..."
412,Breathy,Breathy,"[0.3085710406303406, 0.6310077905654907, 0.132..."
181,Pressed,Pressed,"[0.16483867168426514, 0.5255987048149109, 0.02..."


In [30]:
results_df[results_df['Actual_Label']!=results_df['Predicted_Label']]

,Actual_Label,Predicted_Label,Features
479,Normal,Breathy,"[0.2036820650100708, 0.5917807817459106, 0.171..."
446,Breathy,Pressed,"[0.1821623593568802, 0.6138148903846741, 0.112..."
576,Normal,Breathy,"[0.25455325841903687, 0.572446346282959, 0.151..."
342,Breathy,Pressed,"[0.22416679561138153, 0.504161536693573, 0.127..."
231,Pressed,Breathy,"[0.2095051407814026, 0.6910457611083984, 0.165..."
756,Flow,Pressed,"[0.25982820987701416, 0.5880413055419922, 0.07..."
275,Pressed,Breathy,"[0.29546836018562317, 0.5168625712394714, 0.28..."
617,Normal,Flow,"[0.2186421900987625, 0.5753768086433411, 0.120..."
718,Flow,Pressed,"[0.16297554969787598, 0.6142287254333496, 0.13..."
728,Flow,Normal,"[0.16142725944519043, 0.7253645658493042, 0.16..."
